# Experiment 13: Optuna Best Config — AdamW + CosineAnnealingLR + Dropout 0.45

**Single variable changed**: HPs switched from E1 defaults to Optuna-discovered best values.
**Held constant**: architecture (DiagnosticCNN), loss (CrossEntropy), data (no augmentation).

## Best config from Phase 12

| HP | Optuna best | Prior default |
|----|:-----------:|:-------------:|
| lr | 0.003322 | 0.001 |
| dropout | 0.454 | 0.3 |
| weight_decay | 1.39e-5 | 0 |
| batch_size | 64 | 64 |
| optimizer | AdamW | Adam |
| scheduler | CosineAnnealingLR | constant/linear |
| epochs | 35 | 15 |

Includes logit bias sweep (same as E9) for direct comparison.

In [1]:
import sys; sys.path.append("..")
import os, torch, torch.nn as nn, torch.optim as optim
import numpy as np
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from src.train_utils import train_one_epoch
from src.eval_utils import evaluate, evaluate_detailed, get_all_probas_and_labels, compute_roc_auc_scores, compute_pr_auc_scores

OUT_DIR = "../outputs/error_analysis/optuna_best"
os.makedirs(OUT_DIR, exist_ok=True)

device = ('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## Dataset — identical to E1

In [2]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_ds = __import__('torchvision').datasets.FashionMNIST(root='../data', train=True, download=True, transform=transform)
test_ds = __import__('torchvision').datasets.FashionMNIST(root='../data', train=False, download=True, transform=transform)
class_names = train_ds.classes

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
print(f'Train: {len(train_loader)} batches  Test: {len(test_loader)} batches')

Train: 938 batches  Test: 40 batches


## Model — DiagnosticCNN with Optuna dropout=0.454

In [3]:
class DiagnosticCNN(nn.Module):
    def __init__(self, dropout=0.454):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(dropout), nn.Linear(128, 10),
        )
    def forward(self, x): return self.net(x)

model = DiagnosticCNN().to(device)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

Params: 140,778


c:\document\Study documents\Deeplearning_Course\.venv\Lib\site-packages\torch\nn\modules\module.py:1369: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  return t.to(


## Training — AdamW, lr=0.0033, CosineAnnealingLR, 35 epochs

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.003322, weight_decay=1.39e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=35)
EPOCHS = 35

train_losses = []
model.train()
for epoch in range(EPOCHS):
    loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(loss)
    scheduler.step()
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}] Loss: {loss:.4f}')

torch.save(model.state_dict(), os.path.join(OUT_DIR, 'model_weights.pth'))
with open(os.path.join(OUT_DIR, 'train_losses.txt'), 'w') as f:
    for l in train_losses: f.write(f'{l}\n')
print(f'Done. Final loss: {train_losses[-1]:.4f}')

Epoch [1/35] Loss: 0.4926
Epoch [5/35] Loss: 0.2164
Epoch [10/35] Loss: 0.1510
Epoch [15/35] Loss: 0.0917
Epoch [20/35] Loss: 0.0494
Epoch [25/35] Loss: 0.0242
Epoch [30/35] Loss: 0.0123
Epoch [35/35] Loss: 0.0088
Done. Final loss: 0.0088


## Evaluation (bias=0)

In [5]:
accuracy, cm, per_class = evaluate_detailed(model, test_loader, device, class_names, model_name='OptunaCNN')
cm_np = cm.cpu().numpy()

with open(os.path.join(OUT_DIR, 'metrics_summary.txt'), 'w') as f:
    f.write(f'Test Accuracy (percentage): {accuracy:.2f}\n')
    f.write(f'\n{"Class":<15} {"TPR":>8} {"Precision":>10}\n')
    f.write('-' * 33 + '\n')
    for i, name in enumerate(class_names):
        tpr = per_class[name]['TPR']; prec = per_class[name]['Precision']
        f.write(f'{name:<15} {tpr:>8.4f} {prec:>10.4f}\n')

with open(os.path.join(OUT_DIR, 'confusion_matrix.txt'), 'w') as f:
    f.write(f'{"":>15}')
    for name in class_names: f.write(f'{name:>15}')
    f.write('\n')
    for i in range(10):
        f.write(f'{class_names[i]:>15}')
        for j in range(10): f.write(f'{cm_np[i, j]:>15}')
        f.write('\n')

with open(os.path.join(OUT_DIR, 'misclassification_analysis.txt'), 'w') as f:
    f.write('Misclassification Analysis\n' + '=' * 70 + '\n\n')
    for c in range(10):
        name = class_names[c]; errors = cm_np[c].sum() - cm_np[c, c]
        f.write(f'True: {name} (errors: {errors})\n' + '-' * 50 + '\n')
        for p in np.argsort(-cm_np[c]):
            if p == c or cm_np[c, p] == 0: continue
            f.write(f'  -> {class_names[p]:<15} count={cm_np[c, p]:>4}\n')
        f.write('\n')

print(f'Bias=0: Acc={accuracy:.2f}%  Shirt TPR={per_class["Shirt"]["TPR"]:.4f}  Prec={per_class["Shirt"]["Precision"]:.4f}')

  Test Accuracy: 93.13%
  Class           TPR(Recall)        FPR  Precision
  ---------------------------------------------
  T-shirt/top         0.8900     0.0149     0.8691
  Trouser             0.9790     0.0007     0.9939
  Pullover            0.8980     0.0106     0.9043
  Dress               0.9390     0.0083     0.9260
  Coat                0.9150     0.0126     0.8901
  Sandal              0.9880     0.0012     0.9890
  Shirt               0.7640     0.0212     0.8000
  Sneaker             0.9790     0.0033     0.9703
  Bag                 0.9890     0.0012     0.9890
  Ankle boot          0.9720     0.0023     0.9789
Bias=0: Acc=93.13%  Shirt TPR=0.7640  Prec=0.8000


## Logit bias sweep (same as E9)

In [6]:
SHIRT_IDX = 6
BIASES = [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]
sweep = []

model.eval()
with torch.no_grad():
    for bias in BIASES:
        all_p, all_l = [], []
        sh_tp = sh_fp = sh_fn = 0
        for inputs, lbls in test_loader:
            inputs, lbls = inputs.to(device), lbls.to(device)
            logits = model(inputs)
            logits[:, SHIRT_IDX] += bias
            preds = logits.argmax(dim=1)
            all_p.extend(preds.cpu().numpy()); all_l.extend(lbls.cpu().numpy())
            for true, pred in zip(lbls.cpu().numpy(), preds.cpu().numpy()):
                if pred == SHIRT_IDX and true == SHIRT_IDX: sh_tp += 1
                if pred == SHIRT_IDX and true != SHIRT_IDX: sh_fp += 1
                if pred != SHIRT_IDX and true == SHIRT_IDX: sh_fn += 1
        from sklearn.metrics import accuracy_score
        acc = accuracy_score(all_l, all_p)
        sweep.append({'bias': bias, 'acc': round(acc*100, 2), 'tpr': round(sh_tp/(sh_tp+sh_fn+1e-8), 4), 'prec': round(sh_tp/(sh_tp+sh_fp+1e-8), 4)})
        print(f'bias={bias:+.1f}  acc={acc*100:.2f}%  Shirt TPR={sh_tp/(sh_tp+sh_fn+1e-8):.4f}  Prec={sh_tp/(sh_tp+sh_fp+1e-8):.4f}')

with open(os.path.join(OUT_DIR, 'bias_sweep_results.txt'), 'w') as f:
    f.write(f'{"Bias":>6} {"Acc%":>7} {"ShirtTPR":>9} {"ShirtPrec":>10}\n' + '-' * 32 + '\n')
    for r in sweep:
        f.write(f'{r["bias"]:>+5.1f} {r["acc"]:>7.2f} {r["tpr"]:>9.4f} {r["prec"]:>10.4f}\n')

bias=-1.0  acc=93.14%  Shirt TPR=0.7370  Prec=0.8244
bias=-0.5  acc=93.09%  Shirt TPR=0.7500  Prec=0.8082
bias=+0.0  acc=93.13%  Shirt TPR=0.7640  Prec=0.8000
bias=+0.5  acc=93.07%  Shirt TPR=0.7870  Prec=0.7800
bias=+1.0  acc=93.05%  Shirt TPR=0.8110  Prec=0.7629
bias=+1.5  acc=93.00%  Shirt TPR=0.8250  Prec=0.7480
bias=+2.0  acc=92.81%  Shirt TPR=0.8340  Prec=0.7277


## Comparison vs E9 (prior best) and E1 baseline

In [7]:
bt = max(sweep, key=lambda r: r['acc'] + r['tpr'] * 100)
z = [r for r in sweep if r['bias'] == 0.0][0]

print(f'{"Config":<30} {"Acc%":>7} {"ShirtTPR":>9} {"ShirtPrec":>10}')
print('-' * 56)
print(f'{"E1 CE baseline":<30} {"92.50":>7} {"0.8470":>9} {"0.7227":>10}')
print(f'{"E9 Adam bias=+1.0":<30} {"93.17":>7} {"0.8110":>9} {"0.7776":>10}')
print(f'{"E9 Adam bias=+2.0":<30} {"93.00":>7} {"0.8340":>9} {"0.7507":>10}')
print(f'{"E13 bias=0 (AdamW)":<30} {z["acc"]:>7.2f} {z["tpr"]:>9.4f} {z["prec"]:>10.4f}')
print(f'{"E13 bias="+str(bt["bias"])+" (best trade)":<30} {bt["acc"]:>7.2f} {bt["tpr"]:>9.4f} {bt["prec"]:>10.4f}')

Config                            Acc%  ShirtTPR  ShirtPrec
--------------------------------------------------------
E1 CE baseline                   92.50    0.8470     0.7227
E9 Adam bias=+1.0                93.17    0.8110     0.7776
E9 Adam bias=+2.0                93.00    0.8340     0.7507
E13 bias=0 (AdamW)               93.13    0.7640     0.8000
E13 bias=2.0 (best trade)        92.81    0.8340     0.7277
